In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd

train_data = pd.read_csv('csv/train_dataset.csv')
validation_data = pd.read_csv('csv/val_dataset.csv')
test_data = pd.read_csv('csv/test_dataset.csv')
train_data

In [ ]:
import html
import re

# Function to preprocess text
def preprocess_text(text):
    # Unescape HTML characters
    text = html.unescape(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Function to preprocess text
def preprocess_text2(text):
    # Unescape HTML characters
    text = html.unescape(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '<url>', text)
    # Remove mentions
    text = re.sub(r'@\w+', '<user>', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '<hashtag>', text)
    # Remove special characters and numbers except < and >
    text = re.sub(r'[^a-zA-Z\s<>]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing to train, test, and validation datasets
train_data['Clean'] = train_data['Text'].apply(preprocess_text)
test_data['Clean'] = test_data['Text'].apply(preprocess_text)
validation_data['Clean'] = validation_data['Text'].apply(preprocess_text)

# Apply preprocessing to train, test, and validation datasets
train_data['Clean2'] = train_data['Text'].apply(preprocess_text2)
test_data['Clean2'] = test_data['Text'].apply(preprocess_text2)
validation_data['Clean2'] = validation_data['Text'].apply(preprocess_text2)

# Display a sample of the cleaned text
train_data[['Text', 'Clean', 'Clean2']]

In [ ]:
from nltk.tokenize import TweetTokenizer

tknzr = TweetTokenizer(preserve_case=False)

def tokenize_text(text):
    return tknzr.tokenize(text)

train_data['Tokenized'] = train_data['Clean'].apply(tokenize_text)
test_data['Tokenized'] = test_data['Clean'].apply(tokenize_text)
validation_data['Tokenized'] = validation_data['Clean'].apply(tokenize_text)

train_data['Tokenized2'] = train_data['Clean2'].apply(tokenize_text)
test_data['Tokenized2'] = test_data['Clean2'].apply(tokenize_text)
validation_data['Tokenized2'] = validation_data['Clean2'].apply(tokenize_text)

train_data[['Text', 'Tokenized', 'Tokenized2']]

In [ ]:
import os

if 'glove.twitter.27B.zip' not in os.listdir():
    !wget https://nlp.stanford.edu/data/glove.twitter.27B.zip
else:
    print('Embeddings already downloaded!')
if 'glove.twitter.27B.50d.txt' not in os.listdir():
    !unzip glove*.zip
else:
    print('Embeddings already unzipped!')

In [ ]:
from gensim.models import KeyedVectors

glove_input_file = 'glove.twitter.27B.25d.txt'

model = KeyedVectors.load_word2vec_format(glove_input_file, binary=False, no_header=True)

def index_tweet_tokens(tokens):
    indices = []
    for token in tokens:
        if token in model.key_to_index:
            indices.append(model.key_to_index[token])
        else:
            indices.append(0)
    return indices

train_data['Indices'] = train_data['Tokenized'].apply(index_tweet_tokens)
test_data['Indices'] = test_data['Tokenized'].apply(index_tweet_tokens)
validation_data['Indices'] = validation_data['Tokenized'].apply(index_tweet_tokens)

train_data['Indices2'] = train_data['Tokenized2'].apply(index_tweet_tokens)
test_data['Indices2'] = test_data['Tokenized2'].apply(index_tweet_tokens)
validation_data['Indices2'] = validation_data['Tokenized2'].apply(index_tweet_tokens)


train_data[['Text', 'Tokenized', 'Indices']]

In [ ]:
def count_zero_embeddings(indices):
    count = 0
    for index in indices:
        if index == 0:
            count += 1
    return count

# old pre-processing
train_zero_embeddings = train_data['Indices'].apply(count_zero_embeddings)
validation_zero_embeddings = validation_data['Indices'].apply(count_zero_embeddings)
test_zero_embeddings = test_data['Indices'].apply(count_zero_embeddings)

zero_embed_data = pd.DataFrame({
    'value': train_zero_embeddings.tolist() + validation_zero_embeddings.tolist() + test_zero_embeddings.tolist(),
    'dataset': ['train'] * len(train_zero_embeddings) + ['val'] * len(validation_zero_embeddings) + ['test'] * len(test_zero_embeddings)
})
sns.histplot(zero_embed_data, x='value', hue='dataset', stat='probability', binwidth=1, common_norm=False)
plt.show()

# new pre-processing
train_zero_embeddings = train_data['Indices2'].apply(count_zero_embeddings)
validation_zero_embeddings = validation_data['Indices2'].apply(count_zero_embeddings)
test_zero_embeddings = test_data['Indices2'].apply(count_zero_embeddings)

zero_embed_data = pd.DataFrame({
    'value': train_zero_embeddings.tolist() + validation_zero_embeddings.tolist() + test_zero_embeddings.tolist(),
    'dataset': ['train'] * len(train_zero_embeddings) + ['val'] * len(validation_zero_embeddings) + ['test'] * len(test_zero_embeddings)
})
sns.histplot(zero_embed_data, x='value', hue='dataset', stat='probability', binwidth=1, common_norm=False)
plt.show()

In [ ]:
import torch
from torch import nn


vocab = model.index_to_key
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
weights_matrix = torch.zeros((len(vocab), model.vector_size))
for i, word in enumerate(vocab):
    weights_matrix[i] = torch.tensor(model[word])

embedding_layer = nn.Embedding.from_pretrained(weights_matrix, freeze=False)
embedding_layer

In [ ]:
embedding_layer(torch.tensor([model.key_to_index['hi']]))

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


class TweetDataset(Dataset):
    def __init__(self, tweets, labels, word_to_idx):
        self.tokenizer = TweetTokenizer(preserve_case=False)
        self.tweets = tweets
        self.labels = labels
        self.word_to_idx = word_to_idx

    def __len__(self):
        return len(self.tweets)

    def __getitem__(self, idx):
        tweet = self.tweets[idx]
        label = self.labels[idx]

        tokens = self.tokenizer.tokenize(tweet)
        indices = [self.word_to_idx.get(token, 0) for token in tokens]
        return torch.tensor(indices), torch.tensor(label)

def pad_batch(batch):
    inputs, labels = zip(*batch)
    inputs = pad_sequence(inputs, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return inputs, labels

In [ ]:
train_dataset = TweetDataset(train_data['Clean'], train_data['Label'], word_to_idx)
train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=pad_batch, shuffle=True)
val_dataset = TweetDataset(validation_data['Clean'], validation_data['Label'], word_to_idx)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=pad_batch)

train_dataset2 = TweetDataset(train_data['Clean2'], train_data['Label'], word_to_idx)
train_loader2 = DataLoader(train_dataset2, batch_size=64, collate_fn=pad_batch, shuffle=True)
val_dataset2 = TweetDataset(validation_data['Clean2'], validation_data['Label'], word_to_idx)
val_loader2 = DataLoader(val_dataset2, batch_size=64, collate_fn=pad_batch)

In [ ]:
for inputs, labels in train_dataset:
    print(inputs.shape)
    print(labels.shape)
    break

In [ ]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
import torch.optim as optim

class TweetClassifier(nn.Module):
    def __init__(self, embedding_layer, embedding_dim, hidden_dim, output_dim):
        super(TweetClassifier, self).__init__()
        self.embedding = embedding_layer
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)  # Only 1 output neuron

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)  # (batch_size, seq_len, embedding_dim)
        pooled = embedded.mean(dim=1)          # (batch_size, embedding_dim)
        x = torch.relu(self.fc1(pooled))
        output = self.fc2(x)
        return output  # No activation here, we apply sigmoid in the loss function

# Instantiate model
embedding_dim = model.vector_size
hidden_dim = 128
output_dim = 1  # e.g., 2 classes: positive / negative



In [ ]:
def train_and_validate(model, train_loader, val_loader, criterion, optimizer, device, n_epochs=5):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    model.to(device)

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.squeeze(), labels.float())
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            preds = (outputs.squeeze() > 0.5).float()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_accuracy = train_correct / train_total
        train_losses.append(train_loss / len(train_loader))
        train_accuracies.append(train_accuracy)

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels.float())

                val_loss += loss.item()

                preds = (outputs.squeeze() > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_accuracy = val_correct / val_total
        val_losses.append(val_loss / len(val_loader))
        val_accuracies.append(val_accuracy)

        print(f'Epoch {epoch+1}:')
        print(f'Train Loss: {train_loss/len(train_loader):.4f}, Train Accuracy: {train_accuracy:.4f}')
        print(f'Val Loss: {val_loss/len(val_loader):.4f}, Val Accuracy: {val_accuracy:.4f}')

    return train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
def plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label='Training Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.plot(epochs, train_accuracies, label='Training Accuracy')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Loss/Accuracy')
    plt.title('Training and Validation Metrics')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
mlp1 = TweetClassifier(embedding_layer, embedding_dim, hidden_dim, output_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp1.parameters(), lr=1e-3)

results1 = train_and_validate(mlp1, train_loader, val_loader, criterion, optimizer, device, n_epochs=5)
plot_metrics(*results1)

In [ ]:
mlp2 = TweetClassifier(embedding_layer, embedding_dim, hidden_dim, output_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp2.parameters(), lr=1e-3)

results2 = train_and_validate(mlp2, train_loader2, val_loader2, criterion, optimizer, device, n_epochs=5)
plot_metrics(*results2)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, hidden_dim3):
        """
        input_dim: dimension of input embeddings
        hidden_dim1, hidden_dim2, hidden_dim3: sizes of the three hidden layers
        """
        super(MLP, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, hidden_dim3)
        self.output_layer = nn.Linear(hidden_dim3, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.output_layer(x)
        x = self.sigmoid(x)
        return x

mlp3 = MLP(embedding_layer, embedding_dim, hidden_dim, output_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp3.parameters(), lr=1e-3)

results3 = train_and_validate(mlp3, train_loader2, val_loader2, criterion, optimizer, device, n_epochs=5)
plot_metrics(*results3)

In [ ]:
import optuna
from sklearn.metrics import accuracy_score

def objective(trial):
    # Define the search space for hyperparameters
    hidden_dim1 = trial.suggest_int("hidden_dim1", 32, 256)
    hidden_dim2 = trial.suggest_int("hidden_dim2", 32, 256)
    hidden_dim3 = trial.suggest_int("hidden_dim3", 32, 256)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Create the model with suggested hyperparameters
    model = MLP(embedding_dim, hidden_dim1, hidden_dim2, hidden_dim3).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Train the model
    train_losses, val_losses, train_accuracies, val_accuracies = train_and_validate(
        model, train_loader2, val_loader2, criterion, optimizer, device, n_epochs=5
    )

    # Evaluate the model on the validation set
    val_accuracy = val_accuracies[-1]

    return val_accuracy

# Create an Optuna study
study = optuna.create_study(direction="maximize")  # Maximize validation accuracy

# Optimize the objective function
study.optimize(objective, n_trials=10)  # Adjust the number of trials as needed

# Print the best hyperparameters and validation accuracy
print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))
